# 생성 → 선택 → 학습 통합 노트북

한 노트북 안에서 이미지 생성, 마음에 드는 것 선택, 즉시 학습까지 완결합니다.

## 사용법
1. **셀 1** 실행 (환경 설치 + 모델 로드, 한 번만)
2. **셀 2** 에서 프롬프트/레퍼런스 URL 설정 후 실행 → 이미지 생성
3. 결과를 보고 **셀 3** 에서 번호 입력 → 즉시 학습
4. 프롬프트를 바꿔서 **셀 2~3 반복** → 데이터/학습 누적

## 특징
- 선택한 이미지는 Drive에 자동 저장 (나중에 재학습 가능)
- 학습은 누적됨 (반복할수록 더 정교해짐)
- 언제든 중단해도 Drive에 저장된 LoRA 유지


In [ ]:
# ════════════════════════════════════════════
# 셀 1: 환경 설치 + 모델 로드 (처음 한 번만 실행)
# ════════════════════════════════════════════
!pip install -q diffusers==0.30.0 transformers accelerate peft
!pip install -q bitsandbytes Pillow torch torchvision

import io, json, time, requests
import torch
from pathlib import Path
from PIL import Image
from IPython.display import display, clear_output
from google.colab import drive, userdata
from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image, DDPMScheduler
from peft import LoraConfig, get_peft_model
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from torchvision import transforms

# Google Drive 마운트
drive.mount('/content/drive')

# GitHub 설정
GITHUB_REPO  = "stockpipeline/stockpipeline"
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

# config 로드
cfg      = requests.get(f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/config.json",
                        headers={"Authorization": f"Bearer {GITHUB_TOKEN}"}).json()
img_cfg  = cfg.get("image", {})
NEG      = img_cfg.get("negative_prompt",
           "multiple bowls, many small dishes, tiled pattern, collage")

# 작업 경로
DATASET_DIR = Path("/content/drive/MyDrive/stockpipeline/lora_dataset")
LORA_DIR    = Path("/content/drive/MyDrive/stockpipeline/lora_output/korean_food_lora")
WORK_DIR    = Path("/content/lora_work")
DATASET_DIR.mkdir(parents=True, exist_ok=True)
LORA_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# SDXL-Turbo 로드
MODEL_ID = "stabilityai/sdxl-turbo"
print("모델 로드 중... (2~3분)")
pipe_t2i = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, variant="fp16"
).to("cuda")

# 기존 LoRA가 있으면 자동 로드
lora_loaded = False
if (LORA_DIR / "adapter_model.safetensors").exists():
    try:
        pipe_t2i.load_lora_weights(str(LORA_DIR))
        pipe_t2i.fuse_lora(lora_scale=0.8)
        lora_loaded = True
        print(f"✅ 기존 LoRA 로드됨: {LORA_DIR}")
    except Exception as e:
        print(f"⚠️  LoRA 로드 실패: {e}")

pipe_i2i = AutoPipelineForImage2Image.from_pipe(pipe_t2i)

# LoRA 학습 설정
LORA_RANK     = 16
LEARNING_RATE = 1e-4
STEPS_PER_BATCH = 50  # 이미지 1장당 학습 스텝 수

# 전역 변수 초기화
generated_images = []
total_trained    = len(list(DATASET_DIR.glob("*.jpg")))

print(f"\n✅ 준비 완료 (LoRA: {'적용됨' if lora_loaded else '없음'})")
print(f"   현재 데이터셋: {total_trained}장")
print(f"   이제 셀 2를 실행하세요")


In [ ]:
# ════════════════════════════════════════════
# 셀 2: 이미지 생성 (반복 실행 가능)
# ════════════════════════════════════════════

# ── 여기를 바꾸세요 ──────────────────────────────────────
REF_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Yukgaejang.jpg/1200px-Yukgaejang.jpg"

PROMPT = (
    "a single bowl of Korean yukgaejang spicy beef soup, "
    "shredded beef and fernbrake in red broth, "
    "top-down view, professional food photography, white background"
)

NUM_IMAGES = 6      # 한 번에 생성할 장수
STEPS      = 4
GUIDANCE   = 1.5
STRENGTH   = 0.5    # img2img 강도 (0.3~0.7)
# ────────────────────────────────────────────────────────

# 레퍼런스 이미지 로드
try:
    res     = requests.get(REF_URL, timeout=10)
    ref_img = Image.open(io.BytesIO(res.content)).convert("RGB").resize((1024, 1024))
    print(f"레퍼런스 이미지 로드됨")
    display(ref_img.resize((200, 200)))
except Exception as e:
    ref_img = None
    print(f"⚠️  레퍼런스 로드 실패: {e} → text2img로 전환")

print(f"\n이미지 생성 중... ({NUM_IMAGES}장)")
print("-" * 50)

generated_images = []
start = time.time()

for i in range(NUM_IMAGES):
    seed      = torch.randint(0, 2**31 - 1, (1,)).item()
    generator = torch.Generator(device="cuda").manual_seed(seed)

    if ref_img:
        image = pipe_i2i(
            prompt=PROMPT, image=ref_img,
            negative_prompt=NEG,
            num_inference_steps=STEPS, strength=STRENGTH,
            guidance_scale=GUIDANCE, generator=generator,
        ).images[0]
    else:
        image = pipe_t2i(
            prompt=PROMPT, negative_prompt=NEG,
            num_inference_steps=STEPS, guidance_scale=GUIDANCE,
            width=1024, height=1024, generator=generator,
        ).images[0]

    generated_images.append({
        "image": image, "seed": seed,
        "prompt": PROMPT, "negative_prompt": NEG,
        "steps": STEPS, "guidance": GUIDANCE,
        "strength": STRENGTH, "ref_url": REF_URL,
    })

    print(f"  [{i}] seed={seed}")
    display(image.resize((300, 300)))

print(f"\n완료! {time.time()-start:.1f}초")
print(f"마음에 드는 이미지 번호를 셀 3의 KEEP에 입력하세요 (예: KEEP = [0, 2, 4])")


In [ ]:
# ════════════════════════════════════════════
# 셀 3: 선택 → 즉시 학습
# ════════════════════════════════════════════

# ── 마음에 드는 이미지 번호 입력 ──────────────────
KEEP = [0, 1]   # ← 여기에 번호 입력 (빈 리스트면 저장/학습 안 함)
# ──────────────────────────────────────────────────

if not KEEP:
    print("KEEP이 비어있습니다. 번호를 입력하고 다시 실행하세요.")
else:
    # 1) 선택된 이미지 Drive에 저장
    print(f"선택된 {len(KEEP)}장 저장 중...")
    saved_paths = []

    for idx in KEEP:
        if idx >= len(generated_images):
            print(f"  ⚠️  [{idx}] 범위 초과")
            continue

        entry = generated_images[idx]
        base  = f"lora_train_{len(list(DATASET_DIR.glob('*.jpg')))+1:04d}"

        # 이미지
        img_path = DATASET_DIR / f"{base}.jpg"
        entry["image"].save(str(img_path), "JPEG", quality=95)

        # 캡션 (.txt)
        (DATASET_DIR / f"{base}.txt").write_text(entry["prompt"])

        # 메타데이터 (.json)
        meta = {k: v for k, v in entry.items() if k != "image"}
        (DATASET_DIR / f"{base}.json").write_text(
            json.dumps(meta, ensure_ascii=False, indent=2))

        saved_paths.append(img_path)
        print(f"  [{idx}] 저장: {base}.jpg (seed={entry['seed']})")

    total_trained = len(list(DATASET_DIR.glob("*.jpg")))
    print(f"\n데이터셋 총 {total_trained}장")

    # 2) 즉시 학습
    print(f"\n학습 시작... ({len(saved_paths)}장 × {STEPS_PER_BATCH} steps)")
    print("-" * 50)

    # LoRA 설정 (처음이거나 없으면 새로 생성)
    if not hasattr(pipe_t2i.unet, 'peft_config'):
        lora_config = LoraConfig(
            r=LORA_RANK, lora_alpha=LORA_RANK * 2,
            target_modules=["to_q", "to_k", "to_v", "to_out.0",
                             "add_q_proj", "add_k_proj", "add_v_proj"],
            lora_dropout=0.05, bias="none",
        )
        pipe_t2i.unet = get_peft_model(pipe_t2i.unet, lora_config)

    noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
    optimizer = AdamW(pipe_t2i.unet.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scaler    = GradScaler()

    pipe_t2i.text_encoder.requires_grad_(False)
    pipe_t2i.text_encoder_2.requires_grad_(False)
    pipe_t2i.vae.requires_grad_(False)
    pipe_t2i.unet.train()

    transform = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])

    train_start = time.time()
    losses = []

    for step in range(STEPS_PER_BATCH * len(saved_paths)):
        img_path = saved_paths[step % len(saved_paths)]
        txt_path = img_path.with_suffix('.txt')
        caption  = txt_path.read_text().strip() if txt_path.exists() else PROMPT

        img_t = transform(Image.open(img_path).convert("RGB")).unsqueeze(0).to("cuda", dtype=torch.float16)

        with torch.no_grad():
            def enc(tok, te, texts):
                t = tok(texts, padding="max_length", max_length=tok.model_max_length,
                        truncation=True, return_tensors="pt").input_ids.to("cuda")
                return te(t, output_hidden_states=True)

            o1 = enc(pipe_t2i.tokenizer, pipe_t2i.text_encoder, [caption])
            o2 = enc(pipe_t2i.tokenizer_2, pipe_t2i.text_encoder_2, [caption])
            text_emb   = torch.cat([o1.hidden_states[-2], o2.hidden_states[-2]], dim=-1)
            pooled_emb = o2[0]

            latents = pipe_t2i.vae.encode(img_t).latent_dist.sample()
            latents = latents * pipe_t2i.vae.config.scaling_factor

        noise     = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                  (1,), device="cuda").long()
        noisy     = noise_scheduler.add_noise(latents, noise, timesteps)

        added_cond = {
            "text_embeds": pooled_emb,
            "time_ids": torch.tensor([[1024,1024,0,0,1024,1024]],
                                     device="cuda", dtype=torch.float16),
        }

        with autocast():
            pred = pipe_t2i.unet(noisy, timesteps, text_emb,
                                  added_cond_kwargs=added_cond).sample
            loss = torch.nn.functional.mse_loss(pred.float(), noise.float())

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(pipe_t2i.unet.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        losses.append(loss.item())

        if (step + 1) % 10 == 0:
            avg = sum(losses[-10:]) / 10
            ela = time.time() - train_start
            print(f"  step {step+1}/{STEPS_PER_BATCH * len(saved_paths)} | loss={avg:.4f} | {ela:.0f}s")

    # 3) LoRA 저장
    pipe_t2i.unet.save_pretrained(str(LORA_DIR))
    print(f"\n✅ 학습 완료 + 저장됨: {LORA_DIR}")
    print(f"   총 소요: {(time.time()-train_start)/60:.1f}분")
    print(f"   누적 데이터셋: {total_trained}장")
    print(f"\n셀 2를 다시 실행하면 학습된 LoRA가 적용된 결과를 볼 수 있습니다.")
    print(f"프롬프트를 바꿔서 다른 한식도 생성/학습해보세요.")


In [ ]:
# ════════════════════════════════════════════
# 셀 4: 학습 결과 확인 (선택 실행)
# ════════════════════════════════════════════
# LoRA 적용 후 text2img로 품질 변화 확인

TEST_PROMPTS = [
    "a single bowl of Korean yukgaejang, shredded beef in spicy red broth, top-down view, white background",
    "a single bowl of Korean bibimbap, colorful vegetables and egg on rice, top-down view, white background",
    "a single bowl of Korean kimchi-jjigae, kimchi stew with tofu, top-down view, white background",
]

print("현재 LoRA 적용 상태로 text2img 테스트:")
print("-" * 50)

pipe_t2i.unet.eval()
for i, prompt in enumerate(TEST_PROMPTS):
    gen   = torch.Generator(device="cuda").manual_seed(42 + i)
    image = pipe_t2i(
        prompt=prompt, negative_prompt=NEG,
        num_inference_steps=4, guidance_scale=1.5,
        width=1024, height=1024, generator=gen,
    ).images[0]
    print(f"\n[{i}] {prompt[:50]}...")
    display(image.resize((400, 400)))

pipe_t2i.unet.train()
print("\n마음에 드시면 계속 셀 2~3을 반복하세요.")
